# 09 - Real Training (Faster Engineering Flow)

Run training directly in notebook cells with a 2-stage strategy:
1) fast sweep to identify weak slots
2) focused retraining only where needed


## 1) Mount Google Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2) Clone repo + install deps

- Public repo URL is used below.
- If private, replace clone URL with token URL.


In [2]:
%%bash
set -euo pipefail
cd /content
if [ ! -d Heat-wave-backend/.git ]; then
  git clone https://github.com/orbitorls/HeatShield.git Heat-wave-backend
fi
cd /content/Heat-wave-backend
pip install -r requirements-train.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.3 MB/s eta 0:00:00


Cloning into 'Heat-wave-backend'...
Updating files: 100% (886/886), done.


## 3) Set Python path for `app.*` imports


In [ ]:
import os, sys
REPO = "/content/Heat-wave-backend"
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])

# Persist v3 models + Optuna DB on Drive so warm-starts survive Colab restarts.
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/HeatShield")
DRIVE_MODELS = DRIVE_ROOT / "models" / "forecast_v3"
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
LOCAL_MODELS = Path(REPO) / "app" / "models" / "forecast_v3"
LOCAL_MODELS.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_MODELS.exists() and not LOCAL_MODELS.is_symlink():
    import shutil
    for item in LOCAL_MODELS.iterdir():
        target = DRIVE_MODELS / item.name
        if not target.exists():
            shutil.move(str(item), str(target))
    # rmdir() fails when items already existed on Drive (skipped above).
    # rmtree is safe — everything remaining is already backed up on Drive.
    shutil.rmtree(str(LOCAL_MODELS))
if not LOCAL_MODELS.exists():
    os.symlink(str(DRIVE_MODELS), str(LOCAL_MODELS))
print("v3 models persisted at:", DRIVE_MODELS)


## 4) Ingest real data (NASA POWER)


In [4]:
%%bash
set -euo pipefail
cd /content/Heat-wave-backend
python scripts/ingest_nasa_power.py --start 2021-01-01 --end 2026-05-06


2026-05-06 12:03:30,038 INFO Processing station: BKK_01 (2021-01-01 → 2026-05-06)
2026-05-06 12:03:30,038 INFO Fetching NASA POWER: station=BKK_01 2021-01-01 → 2021-03-31
2026-05-06 12:03:31,833 INFO HTTP Request: GET https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=T2M%2CRH2M%2CWS10M%2CPRECTOTCORR%2CALLSKY_SFC_SW_DWN%2CCLOUD_AMT%2CPS&community=RE&longitude=100.6067&latitude=13.9132&start=20210101&end=20210331&format=JSON&time-standard=UTC "HTTP/1.1 200 OK"
2026-05-06 12:03:32,215 INFO Wrote 24 obs: station=BKK_01 date=2021-01-01 (solar=yes)
2026-05-06 12:03:32,219 INFO Wrote 24 obs: station=BKK_01 date=2021-01-02 (solar=yes)
2026-05-06 12:03:32,222 INFO Wrote 24 obs: station=BKK_01 date=2021-01-03 (solar=yes)
2026-05-06 12:03:32,225 INFO Wrote 24 obs: station=BKK_01 date=2021-01-04 (solar=yes)
2026-05-06 12:03:32,228 INFO Wrote 24 obs: station=BKK_01 date=2021-01-05 (solar=yes)
2026-05-06 12:03:32,231 INFO Wrote 24 obs: station=BKK_01 date=2021-01-06 (solar=yes)
2026-0

## 5) Validate data readiness (min 500 rows per station)


In [5]:
from datetime import date
from app.data.stations import STATIONS
from app.data.loaders import read_observations

start = date(2021, 1, 1)
end = date(2026, 5, 6)
min_rows = 500
bad = []

for sid in STATIONS:
    n = len(read_observations(sid, start, end))
    print(f"{sid}: {n}")
    if n < min_rows:
        bad.append((sid, n))

if bad:
    raise RuntimeError(f"Not enough rows for training: {bad}")

print("Data readiness check: OK")


/content/Heat-wave-backend/app/data/loaders.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(frames, ignore_index=True)


BKK_01: 46800


/content/Heat-wave-backend/app/data/loaders.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(frames, ignore_index=True)


CNX_01: 46800


/content/Heat-wave-backend/app/data/loaders.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(frames, ignore_index=True)


KKN_01: 46800


/content/Heat-wave-backend/app/data/loaders.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(frames, ignore_index=True)


HYI_01: 46799
RYG_01: 46800
Data readiness check: OK


/content/Heat-wave-backend/app/data/loaders.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(frames, ignore_index=True)


## 6) Training helper (stream logs + adaptive flags)


In [ ]:
import datetime
import os
import subprocess
from pathlib import Path

REPO = Path("/content/Heat-wave-backend")
LOG_DIR = REPO / "logs" / "train"
LOG_DIR.mkdir(parents=True, exist_ok=True)

help_text = subprocess.check_output(
    ["python", "scripts/train_forecast.py", "--help"],
    text=True,
    stderr=subprocess.STDOUT,
)

def run_train(*, trials, start, end, station=None, horizons=None, run_tag="run",
              workers=1, target_kind="th", device="auto", gate_backend="lightgbm", force=False):
    run_id = f"{run_tag}_{datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')}"
    cmd = [
        "python", "-u", "scripts/train_forecast.py",
        "--trials", str(trials),
        "--start", start,
        "--end", end,
    ]
    if "--run-id" in help_text:
        cmd += ["--run-id", run_id]
    if "--device" in help_text:
        cmd += ["--device", device]
    if "--gate-backend" in help_text:
        cmd += ["--gate-backend", gate_backend]
    if "--target-kind" in help_text:
        cmd += ["--target-kind", target_kind]
    if "--workers" in help_text and workers > 1:
        cmd += ["--workers", str(workers)]
    if station:
        cmd += ["--station", station]
    if horizons:
        cmd += ["--horizons", horizons]
    if force and "--force" in help_text:
        cmd += ["--force"]

    log_path = LOG_DIR / f"{run_id}.log"
    print("Running:", " ".join(cmd))
    print("Log:", log_path)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    with open(log_path, "w", encoding="utf-8") as f:
        p = subprocess.Popen(
            cmd,
            cwd=str(REPO),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        for line in p.stdout:
            print(line, end="")
            f.write(line)
        rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"Training failed: rc={rc}, log={log_path}")
    return run_id, log_path


## 7) Stage 1: fast sweep (h=24, 5 stations parallel, GPU OpenCL, 25 trials, lightgbm gate)

- h=24 only — the safety-critical alert horizon
- `--workers 5` trains all stations concurrently via `ThreadPoolExecutor`
- `--device gpu` — LightGBM OpenCL (T4 supports OpenCL; CUDA requires a custom wheel)
- Hyperband pruner kills poor trials early → 25 high-quality trials ≈ cost of 15 untuned
- Optuna study persists to Drive → warm-start across session restarts
- `--gate-backend lightgbm` — LightGBM gate, 10-20× faster than BRF

In [ ]:
STAGE1_TRIALS = 25
START = "2021-01-01"
END = "2026-05-06"

stage1_run_id, stage1_log = run_train(
    trials=STAGE1_TRIALS,
    start=START,
    end=END,
    horizons="24",
    run_tag="stage1",
    workers=5,
    device="gpu",
    target_kind="th",
    gate_backend="lightgbm",
)
print("Stage1 run_id:", stage1_run_id)


## 8) Stage 2: refine weak slots only (warm-started Optuna, GPU)

- Reads Stage 1 leaderboard; only `not_ready` / `candidate` slots retrained
- Persistent Optuna study already has 15 Stage-1 trials → Stage 2 adds 100 more around the best region
- `--force` applies **only** to the targeted slot so ready slots stay untouched


In [ ]:
import json
from pathlib import Path

runs_root = Path("/content/Heat-wave-backend/logs/eval/runs")
lb_path = runs_root / stage1_run_id / "leaderboard.json"
if not lb_path.exists():
    raise FileNotFoundError(f"leaderboard.json not found: {lb_path}")

rows = json.loads(lb_path.read_text(encoding="utf-8"))
weak = [
    (r["station"], int(r["horizon_h"]))
    for r in rows
    if r.get("skill_score") is None
    or str(r.get("status", "")).lower() in {"not_ready", "candidate"}
]
print(f"Weak slots: {len(weak)} of {len(rows)}")
for sid, h in weak:
    print(f"  {sid} h{h}")

STAGE2_TRIALS = 100
for sid, h in weak:
    print(f"\nRefining {sid} h{h}")
    run_train(
        trials=STAGE2_TRIALS,
        start=START,
        end=END,
        station=sid,
        horizons=str(h),
        run_tag=f"stage2_{sid}_h{h}",
        workers=1,
        device="gpu",
        target_kind="th",
        gate_backend="lightgbm",
        force=True,
    )


## 10) Stage 3 (optional): train h=6 / h=12 for ready stations

Runs only for stations whose h=24 slot is already `ready`. Uses the warm-started Optuna study (60 additional trials). Skip this cell if you only need the h=24 alert horizon.


In [ ]:
# Reload leaderboard to find which stations are now ready.
import json
from pathlib import Path

all_runs = sorted(Path("/content/Heat-wave-backend/logs/eval/runs").iterdir(), key=lambda p: p.name)
all_rows = []
for run_dir in all_runs:
    lb = run_dir / "leaderboard.json"
    if lb.exists():
        all_rows.extend(json.loads(lb.read_text("utf-8")))

latest = {}
for r in all_rows:
    key = (r["station"], int(r["horizon_h"]))
    latest[key] = r

ready_stations = sorted({sid for (sid, h), r in latest.items()
                          if str(r.get("status", "")).lower() == "ready"})
print(f"Ready stations (h=24): {ready_stations}")

STAGE3_TRIALS = 60
for sid in ready_stations:
    print(f"\nTraining h=6 and h=12 for {sid}")
    run_train(
        trials=STAGE3_TRIALS,
        start=START,
        end=END,
        station=sid,
        horizons="6,12",
        run_tag=f"stage3_{sid}",
        workers=1,
        device="gpu",
        target_kind="th",
        gate_backend="lightgbm",
        force=False,
    )


## 11) Package models + copy to Drive

Zip `app/models/forecast_v3/` (which is symlinked to Drive, so this also works as a portable download).


In [ ]:
import shutil, datetime
from pathlib import Path

stamp = datetime.datetime.now(datetime.UTC).strftime("%Y%m%dT%H%M%SZ")
zip_base = Path(f"/content/forecast_v3_{stamp}")
shutil.make_archive(str(zip_base), "zip", str(LOCAL_MODELS))
zip_path = zip_base.with_suffix(".zip")

exports = DRIVE_ROOT / "exports"
exports.mkdir(parents=True, exist_ok=True)
shutil.copy2(zip_path, exports / zip_path.name)

print(f"Zip:   {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.1f} MB)")
print(f"Drive: {exports / zip_path.name}")
print("Download this file from Google Drive to your local machine.")


## 12) Local evaluation instructions

After downloading the zip, run the following on your local Windows machine.


In [ ]:
print(r"""
LOCAL EVALUATION (Windows PowerShell — run in d:\Heat-wave-backend):

1. Download the zip from Google Drive:
   HeatShield/exports/forecast_v3_*.zip

2. Replace local models:
   Remove-Item app\models\forecast_v3 -Recurse -Force -ErrorAction SilentlyContinue
   Expand-Archive .\forecast_v3_*.zip -DestinationPath app\models\forecast_v3

3. Evaluate:
   python scripts/evaluate_model.py

4. Acceptance tests:
   pytest tests/test_forecast_v2_regression.py -v

5. Smoke-test the API:
   uvicorn app.main:app --reload
   curl "http://localhost:8000/forecast?station=BKK_01&horizon=24"
""")


## 13) Show latest leaderboard


In [ ]:
%%bash
cd /content/Heat-wave-backend
LATEST="$(ls -1t logs/eval/runs 2>/dev/null | head -n1)"
if [ -z "${LATEST}" ]; then
  echo "No run directory found yet"
  exit 0
fi
echo "LATEST RUN: ${LATEST}"
cat "logs/eval/runs/${LATEST}/leaderboard.md" || true
